# Shibuya 3DGS — R2 → COLMAP → train

Cloudflare R2 の写真を直接取得し、3D Gaussian Splatting を学習します。
詳細手順: `docs/colab-r2-3dgs.md`

In [ ]:
# ===== 設定 =====
R2_ACCOUNT_ID = "your_account_id"
R2_ACCESS_KEY_ID = "your_access_key"
R2_SECRET_ACCESS_KEY = "your_secret_key"
R2_BUCKET = "shibuya-scramble"
R2_PREFIX = "photos/"
MAX_IMAGES = 150
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/output"
ITERATIONS = 15000

In [ ]:
!pip install -q boto3 plyfile

import boto3
from pathlib import Path
from botocore.config import Config

input_dir = Path(DATA_DIR) / "input"
input_dir.mkdir(parents=True, exist_ok=True)

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
    config=Config(signature_version="s3v4"),
)

keys = []
continuation = None
while True:
    kwargs = {"Bucket": R2_BUCKET, "Prefix": R2_PREFIX, "MaxKeys": 1000}
    if continuation:
        kwargs["ContinuationToken"] = continuation
    resp = s3.list_objects_v2(**kwargs)
    for obj in resp.get("Contents", []):
        k = obj["Key"]
        if k.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            keys.append(k)
    if not resp.get("IsTruncated"):
        break
    continuation = resp.get("NextContinuationToken")

keys = keys[:MAX_IMAGES]
print(f"対象: {len(keys)} 枚")
assert keys, "R2 に画像がありません"

for i, key in enumerate(keys, 1):
    ext = Path(key).suffix.lower() or ".jpg"
    if ext == ".jpeg":
        ext = ".jpg"
    dest = input_dir / f"{i:04d}{ext}"
    s3.download_file(R2_BUCKET, key, str(dest))
    if i % 20 == 0 or i == len(keys):
        print(f"  {i}/{len(keys)}")
print("✅ done", input_dir)

In [ ]:
%cd /content
!git clone --recursive -q https://github.com/graphdeco-inria/gaussian-splatting
%cd /content/gaussian-splatting
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap > /dev/null
!mkdir -p /content/gaussian-splatting/data
!rm -rf /content/gaussian-splatting/data/input
!ln -s /content/data/input /content/gaussian-splatting/data/input
%cd /content/gaussian-splatting
!python convert.py -s /content/gaussian-splatting/data

In [ ]:
%cd /content/gaussian-splatting
!python train.py -s /content/gaussian-splatting/data -m {OUTPUT_DIR} --iterations {ITERATIONS} --save_iterations 7000 {ITERATIONS}
!find {OUTPUT_DIR} -name "*.ply" -print

学習後の `.ply` は左側ファイル欄からダウンロードするか、`docs/colab-r2-3dgs.md` の R2 アップロードセルを使ってください。